In [1]:
import pandas as pd
import sqlite3

In [7]:
# Load raw booking-level dataset
raw = pd.read_csv("../data/raw/hotel_bookings.csv")

# Convert month name to month number
raw["arrival_date"] = pd.to_datetime(
    raw["arrival_date_year"].astype(str) + "-" +
    raw["arrival_date_month"] + "-" +
    raw["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

print("Raw shape:", raw.shape)
raw[["arrival_date_year", "arrival_date_month", 
     "arrival_date_day_of_month", "arrival_date"]].head()

Raw shape: (119390, 33)


,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date
0,2015,July,1,2015-07-01
1,2015,July,1,2015-07-01
2,2015,July,1,2015-07-01
3,2015,July,1,2015-07-01
4,2015,July,1,2015-07-01


In [8]:
conn = sqlite3.connect(":memory:")

raw.to_sql("hotel_bookings", conn, index=False, if_exists="replace")

print("Table successfully loaded into SQLite.")

Table successfully loaded into SQLite.


In [9]:
daily_aggregation_query = """
SELECT
    arrival_date,
    COUNT(*) AS daily_demand,
    AVG(adr) AS avg_price,
    SUM(CASE WHEN is_canceled = 0 THEN 1 ELSE 0 END) AS completed_bookings,
    SUM(CASE WHEN is_canceled = 1 THEN 1 ELSE 0 END) AS cancellations,
    AVG(stays_in_weekend_nights + stays_in_week_nights) AS avg_length_of_stay
FROM hotel_bookings
WHERE arrival_date IS NOT NULL
GROUP BY arrival_date
ORDER BY arrival_date;
"""

In [10]:
daily_aggregated = pd.read_sql_query(daily_aggregation_query, conn)

print("Daily Aggregated Shape:", daily_aggregated.shape)
daily_aggregated.head()

Daily Aggregated Shape: (793, 6)


,arrival_date,daily_demand,avg_price,completed_bookings,cancellations,avg_length_of_stay
0,2015-07-01 00:00:00,122,92.828934,103,19,3.114754
1,2015-07-02 00:00:00,93,82.205484,36,57,3.935484
2,2015-07-03 00:00:00,56,97.183036,37,19,4.285714
3,2015-07-04 00:00:00,88,85.582273,45,43,4.988636
4,2015-07-05 00:00:00,53,100.002642,37,16,5.962264


In [11]:
feature_query = """
SELECT
    arrival_date,
    COUNT(*) AS daily_demand,
    AVG(adr) AS avg_price,

    CAST(strftime('%m', arrival_date) AS INTEGER) AS month,

    CASE 
        WHEN strftime('%w', arrival_date) IN ('0','6') THEN 1
        ELSE 0
    END AS is_weekend

FROM hotel_bookings
GROUP BY arrival_date
ORDER BY arrival_date;
"""

In [13]:
daily_features = pd.read_sql_query(feature_query, conn)

print("Feature Engineered Shape:", daily_features.shape)
daily_features.head()

Feature Engineered Shape: (793, 5)


,arrival_date,daily_demand,avg_price,month,is_weekend
0,2015-07-01 00:00:00,122,92.828934,7,0
1,2015-07-02 00:00:00,93,82.205484,7,0
2,2015-07-03 00:00:00,56,97.183036,7,0
3,2015-07-04 00:00:00,88,85.582273,7,1
4,2015-07-05 00:00:00,53,100.002642,7,1


In [14]:
daily_features.to_csv("../data/processed/daily_aggregated.csv", index=False)

print("Processed dataset saved.")

Processed dataset saved.


In [15]:
conn.close()

**Purpose:**  
Simulate a production-style SQL data engineering workflow for pricing analytics.

**Key Components:**
- Constructed proper `arrival_date` from year/month/day columns.
- Loaded booking-level data into SQLite.
- Executed SQL aggregation queries to compute:
  - Daily demand
  - Average daily rate (ADR)
  - Cancellation counts
  - Month feature
  - Weekend indicator
- Exported processed dataset to `/data/processed/daily_aggregated.csv`.

**Why It Matters:**  
Demonstrates SQL aggregation, feature engineering, and integration between relational data workflows and Python modeling pipelines.